In [2]:
# Ejercicio 1
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Crear DataFrame clásico
df = pd.DataFrame({
    "Barrio": ["La Virginia", "Puerto Mallarino"],
    "Riesgo": ["Alto", "Medio"],
    "Lon": [-75.88, -76.21],
    "Lat": [4.90, 3.95]
})

# 2. Convertir coordenadas en geometrías tipo Point
geometrias = [Point(xy) for xy in zip(df["Lon"], df["Lat"])]

# 3. Crear GeoDataFrame asignando CRS EPSG:4326
gdf = gpd.GeoDataFrame(
    df,
    geometry=geometrias,
    crs="EPSG:4326"
)

# 4. Imprimir tabla espacial
print(gdf)

# 5. Imprimir sistema de referencia
print("\nSistema de referencia:")
print(gdf.crs)

             Barrio Riesgo    Lon   Lat             geometry
0       La Virginia   Alto -75.88  4.90   POINT (-75.88 4.9)
1  Puerto Mallarino  Medio -76.21  3.95  POINT (-76.21 3.95)

Sistema de referencia:
EPSG:4326


In [3]:
# Ejercicio 2

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# =====================================================
# 1. Crear DataFrame clásico
# =====================================================

df = pd.DataFrame({
    "Barrio": ["La Virginia", "Puerto Mallarino"],
    "Riesgo": ["Alto", "Medio"],
    "Lon": [-75.88, -76.21],
    "Lat": [4.90, 3.95]
})

# =====================================================
# 2. Convertir a GeoDataFrame (EPSG:4326)
# =====================================================

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Lon"], df["Lat"]),
    crs="EPSG:4326"
)

print("GeoDataFrame original:")
print(gdf)

# =====================================================
# 3. Proyectar a MAGNA-SIRGAS Origen Nacional (EPSG:9377)
# =====================================================

gdf_proj = gdf.to_crs(epsg=9377)

print("\nGeoDataFrame proyectado:")
print(gdf_proj)

# =====================================================
# 4. Extraer los puntos proyectados
# =====================================================

punto_1 = gdf_proj.loc[gdf_proj["Barrio"] == "La Virginia", "geometry"].values[0]
punto_2 = gdf_proj.loc[gdf_proj["Barrio"] == "Puerto Mallarino", "geometry"].values[0]

# =====================================================
# 5. Crear línea (LineString)
# =====================================================

linea_acueducto = LineString([punto_1, punto_2])

# =====================================================
# 6. Crear buffer de 5 km (5000 metros)
# =====================================================

buffer_5km = gpd.GeoSeries([linea_acueducto], crs="EPSG:9377").buffer(5000)

# =====================================================
# 7. Evaluar si Puerto Mallarino está dentro del buffer
# =====================================================

dentro_buffer = buffer_5km.iloc[0].contains(punto_2)

print("\n¿Puerto Mallarino está dentro del buffer?")
print(dentro_buffer)

GeoDataFrame original:
             Barrio Riesgo    Lon   Lat             geometry
0       La Virginia   Alto -75.88  4.90   POINT (-75.88 4.9)
1  Puerto Mallarino  Medio -76.21  3.95  POINT (-76.21 3.95)

GeoDataFrame proyectado:
             Barrio Riesgo    Lon   Lat                        geometry
0       La Virginia   Alto -75.88  4.90  POINT (4680685.89 2100129.196)
1  Puerto Mallarino  Medio -76.21  3.95  POINT (4643606.836 1995163.71)

¿Puerto Mallarino está dentro del buffer?
True


In [4]:
# Ejercicio 3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

# =====================================================
# 1. Crear datos originales
# =====================================================

df = pd.DataFrame({
    "Barrio": ["La Virginia", "Puerto Mallarino"],
    "Riesgo": ["Alto", "Medio"],
    "Lon": [-75.88, -76.21],
    "Lat": [4.90, 3.95]
})

# =====================================================
# 2. Convertir a GeoDataFrame (EPSG:4326)
# =====================================================

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Lon"], df["Lat"]),
    crs="EPSG:4326"
)

# =====================================================
# 3. Proyectar a EPSG:9377
# =====================================================

gdf_proj = gdf.to_crs(epsg=9377)

# =====================================================
# 4. Extraer puntos proyectados
# =====================================================

punto_inicio = gdf_proj.loc[
    gdf_proj["Barrio"] == "La Virginia",
    "geometry"
].values[0]

punto_final = gdf_proj.loc[
    gdf_proj["Barrio"] == "Puerto Mallarino",
    "geometry"
].values[0]

# =====================================================
# 5. Crear línea del acueducto
# =====================================================

linea_acueducto = LineString([punto_inicio, punto_final])

# =====================================================
# 6. Interpolar punto al 35% del recorrido
#    normalized=True usa proporción de la longitud total
# =====================================================

punto_fallo = linea_acueducto.interpolate(0.35, normalized=True)

# =====================================================
# 7. Extraer coordenadas Este (X) y Norte (Y)
# =====================================================

este = punto_fallo.x
norte = punto_fallo.y

# =====================================================
# 8. Imprimir resultados
# =====================================================

print("Punto del fallo geológico (35% del recorrido):")
print(punto_fallo)

print("\nCoordenadas proyectadas:")
print(f"Este (X): {este:.2f} metros")
print(f"Norte (Y): {norte:.2f} metros")

Punto del fallo geológico (35% del recorrido):
POINT (4667708.22139359 2063391.2758046228)

Coordenadas proyectadas:
Este (X): 4667708.22 metros
Norte (Y): 2063391.28 metros
